# RAG — Retrieval Augmented Generation

## Jak sprawić, żeby LLM "wiedział" o rzeczach, których nie ma w jego danych treningowych?

### Skąd przychodzimy?

W warsztacie **Function Calling** zbudowaliśmy `search_presidents()` — wyszukiwarkę po prezydentach Polski.
Działała na dopasowaniu słów kluczowych. Widzieliśmy, że pytania typu *"kto zbierał monety"* lub *"związkowiec"* ją łamią.

Dziś zobaczymy **trzy podejścia** do tego problemu:

| # | Metoda | Jak działa | Skaluje się? |
|---|---|---|---|
| 1 | **Substring search** | `if query in text` | ✓ ale nie rozumie języka |
| 2 | **Context stuffing** | Cały tekst → do LLM-a | ✗ limit tokenów |
| 3 | **RAG (embeddingi)** | Wyszukaj top-K → podaj LLM-owi | ✓ + rozumie język |

### Plan warsztatu

1. **Pokażemy problem**: substring search nie rozumie synonimów
2. **Context stuffing**: wpychamy cały dokument do LLM-a — działa, ale nie skaluje się
3. **Embeddingi**: zamienimy tekst na wektory i pokażemy wyszukiwanie semantyczne
4. **RAG**: skleimy wyszukiwanie z LLM-em — i zobaczymy różnicę!

## 1. Konfiguracja środowiska

**Google Colab / MyBinder?** Odkomentuj i uruchom poniższą komórkę — pobierze potrzebne pliki.

In [ ]:
# import os
# _repo = 'https://raw.githubusercontent.com/NXTRSS/MachineLearningCourse/main'
# for f in ['utils.py', 'prezydenci_polski_biografie.txt', 'prezydenci_polski.md']:
#     if not os.path.exists(f):
#         !curl -sO {_repo}/{f}
# print('Pliki pobrane!')

In [ ]:
from utils import ensure_package

ensure_package("openai")
ensure_package("sentence-transformers", "sentence_transformers")
ensure_package("numpy")

import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from IPython.display import display, Markdown
import textwrap

print("Pakiety załadowane!")

## 2. Połączenie z LLM-em

Poniższa komórka automatycznie szuka działającego LLM-a. Zanim ją uruchomisz:

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px; border-radius:4px;">

**Upewnij się, że LLM jest uruchomiony!**

- **LM Studio** (macOS / Windows / Linux) — otwórz aplikację → załaduj model → zakładka *Local Server* → zielony przycisk *Start Server*
- **Ollama** — otwórz terminal i wpisz `ollama serve`
  - *macOS:* jeśli zainstalowałeś przez `.dmg`, Ollama zwykle startuje automatycznie (ikonka w pasku menu)
  - *Windows:* Ollama działa jako usługa po instalacji — jeśli nie, uruchom ręcznie z menu Start
  - *Linux:* `ollama serve` w terminalu (lub `systemctl start ollama` jeśli zainstalowano przez `curl`)

Szczegóły instalacji: patrz `setup_local_llm.ipynb` lub `docs/LOKALNE_LLM.md`
</div>

**Na zajęciach pracujemy na:** `gemma4:e4b` — szybki, lekki model. Pobierz go w LM Studio!

W domu możesz eksperymentować z większymi modelami: `qwen3.6`, `qwen3.5:9b+`, `gemma4:12b`
<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; margin-top:10px; border-radius:4px;">

**Google Colab?** Nie masz lokalnego LLM-a — wpisz adres serwera prowadzącego w `LECTURER_SERVER` poniżej.
Prowadzący poda URL tunnelu na zajęciach (np. `https://abc123.ngrok-free.app`).
</div>

In [ ]:
from utils import connect_llm, extract_reasoning, print_reasoning, clean_content
from utils import setup_auth_client

# Jeśli nie masz lokalnego LLM-a, wpisz adres serwera prowadzącego (podany na zajęciach):
LECTURER_SERVER = "http://ADRES_SERWERA:PORT"  # ← prowadzący poda na zajęciach

client, instructor_client, MODEL_NAME = connect_llm(
    lecturer_server=LECTURER_SERVER,
    model="gemma-4-e4b",  # ← odkomentuj, by wybrać konkretny model (np. "qwen", "llama")
    # backend="lmstudio",          # ← wymuś backend: "lmstudio", "ollama", "lecturer"
    # backend=["lmstudio", "ollama"],  # ← lub lista — próbuj w kolejności
    # ports=[4141],              # ← dodatkowe porty LM Studio na localhost
)

# Serwer prowadzącego? → poprosi o imię (i hasło jeśli wymagane)
client, instructor_client = setup_auth_client(client, instructor_client, MODEL_NAME)

# connect_llm zwraca DWA klienty:
#   client            → do function calling (LLM wybiera narzędzia)
#   instructor_client → do structured output (LLM odpowiada w formacie Pydantic)

if client:
    print(f"\nKlient LLM gotowy!  Model: {MODEL_NAME}")
    print(f"Instructor:         {'tak' if instructor_client else 'nie'}")
    print()
    print("Mamy DWA klienty:")
    print("  client            → do function calling (LLM wybiera narzędzia)")
    print("  instructor_client → do structured output (LLM odpowiada w formacie Pydantic)")


## 3. Problem: substring search nie rozumie języka

W Function Calling zbudowaliśmy `search_presidents()` — wyszukiwarkę po słowach kluczowych.
Zobaczmy na czym polega jej ograniczenie:

In [ ]:
from pathlib import Path

def load_presidents():
    """Ładuje dane o prezydentach z pliku prezydenci_polski.md"""
    md_path = Path("prezydenci_polski.md")
    if not md_path.exists():
        return []
    text = md_path.read_text(encoding="utf-8")
    presidents = []
    current = {}
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("### ") and not line.startswith("####"):
            if current:
                presidents.append(current)
            current = {"imię": line[4:].strip()}
        elif line.startswith("- **") and ":**" in line:
            key = line.split("**")[1].rstrip(":")
            value = line.split(":** ", 1)[1] if ":** " in line else ""
            current[key.lower()] = value
    if current:
        presidents.append(current)
    return presidents

PREZYDENCI = load_presidents()

def search_presidents(query: str) -> str:
    """
    Najprostsza wyszukiwarka — WSZYSTKIE słowa z zapytania muszą
    wystąpić w tekście o prezydencie. Żadnego scoringu, żadnej
    heurystyki — czysty substring matching.
    (W FC mieliśmy ulepszoną wersję ze scoringiem, tutaj celowo
    wracamy do bazowej żeby pokazać ograniczenia.)
    """
    if not PREZYDENCI:
        return "Brak danych — nie znaleziono pliku prezydenci_polski.md"
    query_lower = query.lower()
    words = query_lower.split()
    wyniki = []
    for p in PREZYDENCI:
        all_text = " ".join(f"{k} {v}" for k, v in p.items()).lower()
        if all(w in all_text for w in words):
            lines = [f"### {p.get('imię', '?')}"]
            for key, val in p.items():
                if key != 'imię':
                    lines.append(f"  - {key}: {val}")
            wyniki.append("\n".join(lines))
    if wyniki:
        return "\n\n".join(wyniki)
    return f"Nie znaleziono wyników dla zapytania: {query}"

print(f"Załadowano {len(PREZYDENCI)} prezydentów z prezydenci_polski.md")

In [ ]:
# Zapytania na których substring search ZAWODZI:
test_queries = [
    ("Nobel",                          "odmiana: 'Nobel' ≠ 'Nobla' w tekście"),
    ("kto zbierał monety",            "semantyczne — brak tych słów w tekście"),
    ("najdłużej rządził",             "synonim: 'rządził' ≠ 'kadencja'"),
    ("związkowiec",                    "w tekście 'NSZZ Solidarność', nie 'związkowiec'"),
    ("wojskowy",                       "w tekście 'Wyższa Szkoła Piechoty', nie 'wojskowy'"),
    ("Duda",                             "to akurat znajdzie — szuka dokładnego nazwiska"),
]

print("Substring search — test:\n")
for query, why in test_queries:
    result = search_presidents(query)
    found = "Nie znaleziono" not in result
    status = "ZNALAZŁ ✓" if found else "NIE ZNALAZŁ ✗"
    print(f'  "{query}"')
    print(f"    → {status}  ({why})")
    if found:
        first_line = result.split("\n")[0]
        print(f"    → {first_line}")
    print()

print("Substring search działa tylko gdy pytamy DOKŁADNYMI słowami z tekstu.")
print("A gdyby LLM mógł sam przeczytać cały dokument...?")

## 4. Context stuffing — wrzuć cały dokument do LLM-a

Najprostsze rozwiązanie: zamiast szukać, **dajmy LLM-owi cały tekst** i niech sam znajdzie odpowiedź.

```
Użytkownik: "Kto zbierał monety?"
    ↓
Prompt: "Oto dane o prezydentach: [CAŁY TEKST]. Odpowiedz na pytanie: ..."
    ↓
LLM: "Andrzej Duda — w tekście jest mowa o numizmatyce"
```

To działa! Ale jest haczyk — **co jeśli dokument ma 1000 stron?** LLM ma limit tokenów (context window).
Zobaczmy jak to wygląda na naszym małym zbiorze danych:

In [ ]:
# === Context stuffing — cały dokument w prompcie ===
import time

_PRESIDENTS_RAW = Path("prezydenci_polski.md").read_text(encoding="utf-8")
print(f"📄 Dokument wczytany raz: {len(_PRESIDENTS_RAW)} znaków."
     " Każde zapytanie wyśle go do LLM-a w całości.\n")

def context_stuffing_search(query: str) -> str:
    """Wysyła CAŁY dokument do LLM-a i pyta o odpowiedź."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content":
             "Odpowiadasz WYŁĄCZNIE na podstawie podanych danych o prezydentach. "
             "Jeśli danych brak — powiedz że nie wiesz. Nie wymyślaj. Odpowiadaj po polsku."},
            {"role": "user", "content":
             f"Dane o prezydentach:\n{_PRESIDENTS_RAW}\n\nPytanie: {query}"}
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content

# Przetestujmy na tych samych zapytaniach co substring:
if client:
    _is_local = "127.0.0.1" in str(client.base_url) or "localhost" in str(client.base_url)
    _queries_to_run = test_queries if _is_local else test_queries[:1]

    if _is_local:
        print(f"Context stuffing — lokalny LLM, odpalam {len(_queries_to_run)} zapytań:\n")
    else:
        print(f"⚠️  Serwer prowadzącego — odpalam 1 z {len(test_queries)} zapytań"
              " (żeby nie przeciążać serwera).\n")

    _times = []
    for query, why in _queries_to_run:
        print(f'  "{query}"')
        _t0 = time.time()
        answer = context_stuffing_search(query)
        _dt = time.time() - _t0
        _times.append(_dt)
        print(f"    → {answer[:150]}")
        print(f"    ⏱  {_dt:.1f}s")
        print()

    if _is_local and len(_times) > 1:
        _avg = sum(_times) / len(_times)
        print(f"Średnio {_avg:.1f}s / zapytanie  →"
              f"  {len(test_queries)} pytań = ~{_avg * len(test_queries):.0f}s łącznie.")

    print(f"\nDziała! Ale nasz dokument ma tylko {len(_PRESIDENTS_RAW)} znaków.")
    print("Co jeśli mamy 100 000 znaków? Albo milion? LLM tego nie pomieści.")
else:
    print("LLM niedostępny — przejdź dalej do embeddingów.")

### Ćwiczenie 1: Porównaj substring vs context stuffing

Uruchom OBA podejścia na własnym pytaniu i porównaj wyniki.
W którym przypadku odpowiedź jest lepsza?

In [ ]:
# === Ćwiczenie 1 ===

MOJE_PYTANIE = "kto miał związek z wojskiem?"  # ← ZMIEŃ NA SWOJE

### Proszę poniżej uzupełnić kod ### (≈ 4 linijki)
# Wskazówka: wywołaj search_presidents() i context_stuffing_search() z MOJE_PYTANIE

wynik_substring = ...  # Tutaj wpisz swój kod
wynik_stuffing = ...   # Tutaj wpisz swój kod

### Koniec uzupełniania kodu ###

try:
    print(f'Pytanie: "{MOJE_PYTANIE}"\n')
    print(f"Substring search:\n  {wynik_substring[:200]}\n")
    print(f"Context stuffing:\n  {wynik_stuffing[:200]}")
except (TypeError, NameError):
    print("⬆️ Uzupełnij kod powyżej")

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Użyj `search_presidents(MOJE_PYTANIE)` dla substring search
i `context_stuffing_search(MOJE_PYTANIE)` dla context stuffing.

Pamiętaj o obsłudze przypadku gdy LLM jest niedostępny (`client` jest `None`).

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# === Ćwiczenie 1 — rozwiązanie ===

MOJE_PYTANIE = "kto miał związek z wojskiem?"

wynik_substring = search_presidents(MOJE_PYTANIE)
wynik_stuffing = context_stuffing_search(MOJE_PYTANIE) if client else "(LLM niedostępny)"

print(f'Pytanie: "{MOJE_PYTANIE}"\n')
print(f"Substring search:\n  {wynik_substring[:200]}\n")
print(f"Context stuffing:\n  {wynik_stuffing[:200]}")

###### 

## 5. Trzy podejścia — porównanie

| | Substring search | Context stuffing | RAG (embeddingi) |
|---|---|---|---|
| **Rozumie synonimy?** | ✗ | ✓ | ✓ |
| **Rozumie kontekst?** | ✗ | ✓ | ✓ |
| **Skaluje się?** | ✓ (szybkie) | ✗ (limit tokenów) | ✓ |
| **Wymaga LLM-a?** | ✗ | ✓ | ✓ (do embeddingów nie) |
| **Koszt** | ~0 | Wysoki (cały dokument) | Niski (tylko top-K) |

Context stuffing to świetne rozwiązanie na małe zbiory danych (do ~10 stron).
Ale gdy mamy **setki lub tysiące** dokumentów — potrzebujemy RAG.

Teraz zbudujemy RAG krok po kroku!

## 6. Przygotowanie dokumentów — chunking

Mamy plik `prezydenci_polski_biografie.txt` z rozbudowanymi biografiami prezydentów Polski.
Zanim zbudujemy RAG, musimy podzielić tekst na **fragmenty (chunks)**.

### Dlaczego dzielimy na fragmenty?
- Modele embeddingowe najlepiej działają na krótszych tekstach (1–3 akapity)
- Chcemy wyszukać **konkretny** fragment, nie cały dokument
- To samo robią profesjonalne systemy RAG (LangChain, LlamaIndex, itd.)

In [ ]:
# === Wczytanie dokumentu ===

with open("prezydenci_polski_biografie.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

_md = open("prezydenci_polski.md", encoding="utf-8").read()

print("Porównanie dokumentów:")
print(f"  prezydenci_polski.md        {len(_md):>7,} znaków  ← używaliśmy w context stuffing")
print(f"  prezydenci_polski_biografie {len(full_text):>7,} znaków  "
      f"← {len(full_text)/len(_md):.0f}× więcej")
print()
print(f"Context stuffing z biografiami → {len(full_text):,} znaków przy każdym zapytaniu.")
print(f"Limit modelu: ~128k tokenów ≈ 500k znaków — biografie zajmują {len(full_text)/500_000*100:.1f}% limitu.")
print(f"Wikipedia? Kodeks cywilny? Miliony znaków → context stuffing odpada. Dlatego RAG.")
print()
print(f"Pierwsze 300 znaków biografii:\n")
print(full_text[:300] + "...")


In [ ]:
# === Podział na fragmenty (chunking) ===

def chunk_text(text, chunk_size=300, overlap=50):
    """
    Dzieli tekst na nakładające się fragmenty.

    Args:
        text: tekst do podziału
        chunk_size: docelowy rozmiar fragmentu (w znakach)
        overlap: ile znaków z końca poprzedniego fragmentu
                 pojawia się na początku następnego

    Dlaczego overlap? Żeby nie "przeciąć" zdania w połowie
    i nie stracić kontekstu na granicy fragmentów.
    """
    # Dzielimy po podwójnych enterach (akapitach)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + 1 + len(para) <= chunk_size:
            current_chunk += (" " if current_chunk else "") + para
        else:
            if current_chunk:
                chunks.append(current_chunk)
                # overlap: nowy chunk zaczyna się od ostatnich N znaków poprzedniego
                tail = current_chunk[-overlap:] if overlap else ""
                current_chunk = (tail + " " + para).strip() if tail else para
            else:
                current_chunk = para

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

chunks = chunk_text(full_text, chunk_size=500, overlap=50)

print(f"Dokument podzielony na {len(chunks)} fragmentów\n")
PREVIEW = 10
for i, chunk in enumerate(chunks[:PREVIEW]):
    print(f"--- Fragment {i+1} ({len(chunk)} znaków) ---")
    print(textwrap.fill(chunk[:150], width=80) + ("..." if len(chunk) > 150 else ""))
    print()

if len(chunks) > PREVIEW:
    print(f"... i jeszcze {len(chunks) - PREVIEW} fragmentów (pokazano {PREVIEW} z {len(chunks)}).")


### Ćwiczenie 2: Eksperymentuj z rozmiarem fragmentów

Spróbuj zmienić parametry `chunk_size` i `overlap` w komórce powyżej.

> **`overlap`** — ile znaków "zachodzi" na siebie między sąsiednimi fragmentami.
> Np. `overlap=50` przy `chunk_size=300` oznacza, że ostatnie 50 znaków fragmentu N
> pojawia się też na początku fragmentu N+1. Dzięki temu zdanie "przecięte" na granicy
> chunka nie ginie — RAG może je znaleźć z obu stron.

**Proszę uzupełnić poniższą komórkę** — uruchom chunking z innymi parametrami i porównaj wyniki:


In [ ]:
# === Ćwiczenie 2 ===
# Spróbuj różnych rozmiarów fragmentów i zobacz jak to wpływa na liczbę chunków

### Proszę poniżej uzupełnić kod ### (≈ 2 linijki)
# Wskazówka: wywołaj chunk_text z innymi parametrami, np. chunk_size=200 lub chunk_size=1000

chunks_male = ...  # Tutaj wpisz swój kod
chunks_duze = ...  # Tutaj wpisz swój kod

### Koniec uzupełniania kodu ###

try:
    print(f"Domyślne (500 znaków): {len(chunks)} fragmentów")
    print(f"Małe (200 znaków):     {len(chunks_male)} fragmentów")
    print(f"Duże (1000 znaków):    {len(chunks_duze)} fragmentów")
    print(f"\nJak myślisz — lepiej mieć więcej małych fragmentów czy mniej dużych? Dlaczego?")
except (TypeError, NameError):
    print("⬆️ Uzupełnij kod powyżej")

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# === Ćwiczenie 2 — rozwiązanie ===

chunks_male = chunk_text(full_text, chunk_size=200, overlap=30)
chunks_duze = chunk_text(full_text, chunk_size=1000, overlap=100)

print(f"Domyślne (500 znaków): {len(chunks)} fragmentów")
print(f"Małe (200 znaków):     {len(chunks_male)} fragmentów")
print(f"Duże (1000 znaków):    {len(chunks_duze)} fragmentów")

print(f"\nPrzykładowy mały fragment:")
print(f"  '{chunks_male[0][:100]}...'")
print(f"\nPrzykładowy duży fragment:")
print(f"  '{chunks_duze[0][:100]}...'")

###### 

## 7. Embeddingi — zamiana tekstu na wektory

Pamiętacie **Word2Vec** z poprzednich zajęć? Tam każde *słowo* miało swój wektor.
Teraz robimy to samo, ale dla **całych fragmentów tekstu** — to są tzw. *sentence embeddings*.

Używamy modelu `paraphrase-multilingual-MiniLM-L12-v2` — lekkiego modelu (do uruchomienia na CPU),
który rozumie wiele języków, w tym **polski**.

### Jak to działa?
```
"Prezydent Kaczyński zginął w katastrofie" → [0.12, -0.45, 0.78, ..., 0.03]  (384 liczby)
```

Teksty o podobnym *znaczeniu* będą miały **bliskie wektory** (mały kąt między nimi = wysokie cosine similarity).

In [ ]:
# === Ładowanie modelu embeddingowego ===
# Ten model działa na CPU — nie potrzeba GPU!
# Przy pierwszym uruchomieniu zostanie pobrany (~120 MB)

print("Ładowanie modelu embeddingowego...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(f"Model załadowany!")
print(f"Wymiar embeddingów: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
# === Tworzenie embeddingów dla naszych fragmentów ===

print(f"Tworzę embeddingi dla {len(chunks)} fragmentów...")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)

print(f"\nGotowe! Kształt macierzy embeddingów: {chunk_embeddings.shape}")
print(f"Każdy fragment to wektor {chunk_embeddings.shape[1]} liczb\n")

# Zobaczmy jak wygląda embedding pierwszego fragmentu
print(f"Embedding fragmentu 1 (pierwsze 10 wartości):")
print(np.round(chunk_embeddings[0][:10], 4))

## 8. Cosine similarity — miara podobieństwa wektorów

### Ćwiczenie 3: Cosine similarity "ręcznie"

Zanim użyjemy gotowej funkcji wyszukiwania, policzmy cosine similarity sami.

Przypomnijmy wzór:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

Gdzie $A \cdot B$ to iloczyn skalarny, a $\|A\|$ to norma (długość) wektora.

In [ ]:
# === Ćwiczenie 3: Policz cosine similarity ręcznie ===

try:
    # Pytanie o Nagrodę Nobla — porównajmy z dwoma fragmentami:
    #   chunks[23] → o Wałęsie i Pokojowej Nagrodzie Nobla (powinien być WYSOKI)
    #   chunks[5]  → o służbie wojskowej Jaruzelskiego (powinien być NISKI)
    pytanie = "Kto otrzymał Pokojową Nagrodę Nobla?"
    emb_pytania = embed_model.encode([pytanie])[0]
    emb_fragment_a = chunk_embeddings[23]   # Wałęsa — Nobel
    emb_fragment_b = chunk_embeddings[5]    # Jaruzelski — wojsko

    ### Proszę poniżej uzupełnić kod ### (≈ 2 linijki)
    # Wskazówka: użyj np.dot() do iloczynu skalarnego i np.linalg.norm() do normy

    similarity_a = ...  # Tutaj wpisz swój kod
    similarity_b = ...  # Tutaj wpisz swój kod

    ### Koniec uzupełniania kodu ###

    print(f"Pytanie: '{pytanie}'\n")
    print(f"Fragment A (Wałęsa — Nobel):")
    print(f"  '{chunks[23][:100]}...'")
    print(f"  Cosine similarity: {similarity_a:.4f}\n")
    print(f"Fragment B (Jaruzelski — wojsko):")
    print(f"  '{chunks[5][:100]}...'")
    print(f"  Cosine similarity: {similarity_b:.4f}\n")
    print(f"Który fragment jest bliższy pytaniu? {'Fragment A ✓' if similarity_a > similarity_b else 'Fragment B'}")
except (TypeError, NameError):
    print("⬆️ Uzupełnij kod powyżej")

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Cosine similarity to iloczyn skalarny podzielony przez iloczyn norm:

$$\text{sim}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

W NumPy: `np.dot(a, b)` daje iloczyn skalarny, `np.linalg.norm(a)` daje normę wektora.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# === Ćwiczenie 3 — rozwiązanie ===

pytanie = "Kto otrzymał Pokojową Nagrodę Nobla?"
emb_pytania = embed_model.encode([pytanie])[0]
emb_fragment_a = chunk_embeddings[23]   # Wałęsa — Nobel
emb_fragment_b = chunk_embeddings[5]    # Jaruzelski — wojsko

similarity_a = np.dot(emb_pytania, emb_fragment_a) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_a))
similarity_b = np.dot(emb_pytania, emb_fragment_b) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_b))

print(f"Pytanie: '{pytanie}'\n")
print(f"Fragment A (Wałęsa — Nobel):")
print(f"  '{chunks[23][:100]}...'")
print(f"  Cosine similarity: {similarity_a:.4f}\n")
print(f"Fragment B (Jaruzelski — wojsko):")
print(f"  '{chunks[5][:100]}...'")
print(f"  Cosine similarity: {similarity_b:.4f}\n")
print(f"Który fragment jest bliższy pytaniu? {'Fragment A ✓' if similarity_a > similarity_b else 'Fragment B'}")
print(f"\nRóżnica: {abs(similarity_a - similarity_b):.4f} — embeddingi wyraźnie 'wiedzą' który tekst pasuje!")

###### 

## 9. Wyszukiwanie semantyczne

Teraz najciekawsza część! Zamieniamy pytanie użytkownika na embedding,
a potem szukamy **najbliższych** fragmentów tekstu.

"Najbliższy" = najwyższe **cosine similarity** (podobieństwo kosinusowe).

To jest *serce* RAG-a — wyszukiwanie semantyczne. Zauważcie, że nie szukamy po słowach kluczowych
(jak w Google), ale po **znaczeniu** tekstu!

In [ ]:
# === Funkcja wyszukiwania semantycznego ===

def search(query, chunks, chunk_embeddings, embed_model, top_k=3):
    """
    Wyszukuje top_k najbardziej pasujących fragmentów do zapytania.
    
    Kroki:
    1. Zamień pytanie na embedding
    2. Oblicz cosine similarity z każdym fragmentem
    3. Zwróć top_k najlepszych
    """
    # 1. Embedding pytania
    query_embedding = embed_model.encode([query])[0]
    
    # 2. Cosine similarity = dot product znormalizowanych wektorów
    #    similarity(A, B) = (A · B) / (||A|| * ||B||)
    similarities = []
    for i, chunk_emb in enumerate(chunk_embeddings):
        dot_product = np.dot(query_embedding, chunk_emb)
        norm_q = np.linalg.norm(query_embedding)
        norm_c = np.linalg.norm(chunk_emb)
        similarity = dot_product / (norm_q * norm_c)
        similarities.append((i, similarity, chunks[i]))
    
    # 3. Sortuj malejąco po similarity
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]

print("Funkcja wyszukiwania gotowa!")

In [ ]:
# === Testujemy wyszukiwanie! ===

query = "Kto i za co otrzymał Pokojową Nagrodę Nobla?"

print(f"Pytanie: {query}")
print("="*70)

results = search(query, chunks, chunk_embeddings, embed_model, top_k=3)

print(f"\nZnaleziono {len(results)} najbardziej trafnych fragmentów:\n")

for rank, (idx, score, chunk_text) in enumerate(results, 1):
    print(f"{'─'*70}")
    print(f"  Trafność #{rank} | Fragment nr {idx+1} | Cosine similarity: {score:.4f}")
    print(f"{'─'*70}")
    # Pokazujemy cały fragment — student widzi CO DOKŁADNIE embeddingi znalazły
    print(textwrap.fill(chunk_text, width=70, initial_indent="  ", subsequent_indent="  "))
    print()

In [ ]:
# === Sprawdźmy inne pytania! ===

semantic_test_queries = [
    "Kto był najmłodszym prezydentem Polski?",
    "Który prezydent wprowadził stan wojenny i kiedy?",
    "Jak długo trwała kadencja Aleksandra Kwaśniewskiego?",
]

for q in semantic_test_queries:
    print(f"\n{'='*70}")
    print(f"PYTANIE: {q}")
    print(f"{'='*70}")
    results = search(q, chunks, chunk_embeddings, embed_model, top_k=2)
    for rank, (idx, score, text) in enumerate(results, 1):
        print(f"\n  #{rank} [similarity: {score:.4f}] Fragment {idx+1}:")
        print(textwrap.fill(text[:200], width=70, initial_indent="  ", subsequent_indent="  "))
        if len(text) > 200:
            print("  ...")

## 10. RAG — łączymy wyszukiwanie z LLM-em!

Teraz kluczowy moment. Mamy:
- **Pytanie** użytkownika
- **Fragmenty** tekstu znalezione przez wyszukiwanie semantyczne

Składamy to razem w **prompt** dla LLM-a:

```
Kontekst (znalezione fragmenty):
[Fragment 1]
[Fragment 2]
[Fragment 3]

Pytanie: ...
Odpowiedz WYŁĄCZNIE na podstawie powyższego kontekstu.
```

To jest cały "sekret" RAG-a — dajemy LLM-owi kontekst, którego sam nie ma!

In [ ]:
# === Budowanie prompta RAG ===

TEST_QUESTION = "Jak długo trwała kadencja Aleksandra Kwaśniewskiego i ile miał lat obejmując urząd?"

def build_rag_prompt(query, retrieved_chunks):
    """
    Buduje prompt RAG z pytaniem i znalezionymi fragmentami.
    """
    context_parts = []
    for rank, (idx, score, text) in enumerate(retrieved_chunks, 1):
        context_parts.append(f"[Fragment {rank} | trafność: {score:.3f}]\n{text}")
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""Na podstawie poniższych fragmentów odpowiedz na pytanie.
Używaj WYŁĄCZNIE informacji z podanych fragmentów. Nie wymyślaj faktów.
Jeśli fragmenty nie zawierają odpowiedzi, napisz dokładnie:
"Brak informacji w dostarczonych fragmentach."

=== FRAGMENTY ===
{context}
=== KONIEC FRAGMENTÓW ===

Pytanie: {query}

Odpowiedź:"""
    
    return prompt

# Przetestujmy jak wygląda gotowy prompt
results = search(TEST_QUESTION, chunks, chunk_embeddings, embed_model, top_k=3)
rag_prompt = build_rag_prompt(TEST_QUESTION, results)

print("=== GOTOWY PROMPT RAG (to idzie do LLM-a) ===")
print(rag_prompt)
print(f"\n\nDługość prompta: {len(rag_prompt)} znaków")

In [ ]:
# === Odpowiedź LLM-a Z RAG-iem ===

# Budujemy prompt od nowa (nie polegamy na zmiennej z innej komórki)
results = search(TEST_QUESTION, chunks, chunk_embeddings, embed_model, top_k=3)
rag_prompt = build_rag_prompt(TEST_QUESTION, results)

if client:
    print(f"Pytanie: {TEST_QUESTION}")
    print(f"Model: {MODEL_NAME}")
    print("="*60)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, krótko i na temat."},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
    )
    
    rag_answer = response.choices[0].message.content
    print(f"\nOdpowiedź LLM-a Z RAG-iem:\n")
    print(textwrap.fill(rag_answer, width=80))
else:
    print("LLM niedostępny. Powyżej widać prompt który normalnie poszedłby do modelu.")

## 11. Porównanie: BEZ RAG vs Z RAG

Zobaczmy różnicę na wielu pytaniach!

In [ ]:
# === Pełne porównanie: bez RAG vs z RAG ===

questions = [
    "Kto był najmłodszym prezydentem Polski i ile miał lat obejmując urząd?",
    "Który prezydent zginął w katastrofie lotniczej i kiedy to się wydarzyło?",
    "Kto i za co otrzymał Pokojową Nagrodę Nobla?",
    # ↓ Pytanie-pułapka: Zaleski był prezydentem na uchodźstwie, nie III RP
    #   RAG powinien odpowiedzieć "Brak informacji" — i to jest POPRAWNA odpowiedź!
    "Jak długo August Zaleski pełnił funkcję prezydenta?",
]

if client:
    for q in questions:
        print(f"\n{'='*70}")
        print(f"PYTANIE: {q}")
        print(f"{'='*70}")
        
        # Bez RAG
        resp_no_rag = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Odpowiadaj krótko i po polsku."},
                {"role": "user", "content": q}
            ],
            temperature=0.1,
        )
        
        # Z RAG
        results = search(q, chunks, chunk_embeddings, embed_model, top_k=3)
        rag_prompt = build_rag_prompt(q, results)
        
        resp_rag = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, krótko i na temat."},
                {"role": "user", "content": rag_prompt}
            ],
            temperature=0.1,
        )
        
        print(f"\n  BEZ RAG: {resp_no_rag.choices[0].message.content[:200]}")
        print(f"\n  Z RAG:   {resp_rag.choices[0].message.content[:200]}")
        
        # Pokaż użyte fragmenty
        print(f"\n  Użyte fragmenty (top {len(results)}):")
        for rank, (idx, score, text) in enumerate(results, 1):
            print(f"    #{rank} [sim={score:.3f}]: {text[:80]}...")
else:
    print("LLM niedostępny — porównanie wymaga modelu.")

## 12. Zadaj własne pytanie!

Teraz Twoja kolej — wpisz pytanie o prezydentach Polski i zobacz jak działa RAG.

In [ ]:
# === Twoje pytanie! ===

TWOJE_PYTANIE = "Ile dni trwała kadencja Wojciecha Jaruzelskiego?"  # <-- ZMIEŃ NA SWOJE!


# --- Wyszukiwanie semantyczne ---
print(f"Pytanie: {TWOJE_PYTANIE}")
print("="*70)

results = search(TWOJE_PYTANIE, chunks, chunk_embeddings, embed_model, top_k=3)

print(f"\nZnalezione fragmenty:\n")
for rank, (idx, score, text) in enumerate(results, 1):
    print(f"  #{rank} [cosine similarity: {score:.4f}]:")
    print(textwrap.fill(text, width=70, initial_indent="    ", subsequent_indent="    "))
    print()

# --- RAG ---
if client:
    rag_prompt = build_rag_prompt(TWOJE_PYTANIE, results)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku."},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
    )
    
    print(f"{'='*70}")
    print(f"Odpowiedź RAG:\n")
    print(textwrap.fill(response.choices[0].message.content, width=70))
else:
    print("LLM niedostępny — ale powyżej widzisz co wyszukiwanie znalazło!")

## 13. Co pod spodem? Wizualizacja embeddingów

Zobaczmy jak nasze fragmenty i pytanie wyglądają w przestrzeni wektorowej.
Zredukujemy 384-wymiarowe embeddingi do 2D za pomocą **PCA** (Principal Component Analysis) — tej samej metody redukcji wymiarów, którą znacie z wcześniejszych zajęć.

In [ ]:
# === Wizualizacja embeddingów (PCA) ===
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Embedding pytania
query_for_viz = "Kto i za co otrzymał Pokojową Nagrodę Nobla?"
query_emb = embed_model.encode([query_for_viz])

# Łączymy embeddingi fragmentów + pytania
all_embeddings = np.vstack([chunk_embeddings, query_emb])

# PCA redukcja do 2D
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(all_embeddings)

print(f"PCA — wyjaśniona wariancja: {pca.explained_variance_ratio_[0]:.1%} + {pca.explained_variance_ratio_[1]:.1%}"
      f" = {sum(pca.explained_variance_ratio_):.1%}\n")

# Rysujemy
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

# Fragmenty dokumentu
chunk_points = embeddings_2d[:-1]
ax.scatter(chunk_points[:, 0], chunk_points[:, 1],
           c='steelblue', s=100, alpha=0.7, label='Fragmenty dokumentu')

# Etykiety fragmentów
for i, (x, y) in enumerate(chunk_points):
    ax.annotate(f'F{i+1}', (x, y), textcoords="offset points",
                xytext=(5, 5), fontsize=7, alpha=0.6)

# Pytanie
query_point = embeddings_2d[-1]
ax.scatter(query_point[0], query_point[1],
           c='red', s=200, marker='*', label='Pytanie', zorder=5)
ax.annotate(f'Q: {query_for_viz[:40]}...',
            (query_point[0], query_point[1]),
            textcoords="offset points", xytext=(10, 10),
            fontsize=9, color='red', fontweight='bold')

# Zaznacz top-3 najbliższe fragmenty
results = search(query_for_viz, chunks, chunk_embeddings, embed_model, top_k=3)
for rank, (idx, score, _) in enumerate(results):
    x, y = chunk_points[idx]
    ax.scatter(x, y, c='orange', s=200, alpha=0.5,
               edgecolors='red', linewidth=2)
    ax.annotate(f'TOP {rank+1}\n(sim={score:.3f})', (x, y),
                textcoords="offset points", xytext=(-15, -20),
                fontsize=8, color='darkorange', fontweight='bold')

ax.set_title('Embeddingi fragmentów dokumentu + pytanie (PCA 2D)', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} wariancji)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} wariancji)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Fragmenty najbliżej pytania (pomarańczowe) to te, które trafiają do RAG prompta!")

## Podsumowanie

### Co zrobiliśmy?

1. **Substring search** — zobaczyliśmy że wyszukiwanie po słowach kluczowych nie rozumie języka
2. **Context stuffing** — wrzuciliśmy cały dokument do LLM-a → działa, ale nie skaluje się
3. **Chunking** — podzieliliśmy tekst na fragmenty
4. **Embeddingi** — zamieniliśmy fragmenty na wektory liczbowe (384 wymiary)
5. **Wyszukiwanie semantyczne** — znaleźliśmy fragmenty najbliższe pytaniu (cosine similarity)
6. **RAG** — połączyliśmy wyszukiwanie z LLM-em → poprawne odpowiedzi!

### Trzy podejścia — kiedy co?

| | Substring | Context stuffing | RAG |
|---|---|---|---|
| **Kiedy?** | Mały, strukturalny zbiór | < 10 stron tekstu | Duże zbiory dokumentów |
| **Przykład** | `search_presidents()` z FC | Cały .md do prompta | Embeddingi + top-K + LLM |

### RAG w produkcji

W prawdziwych zastosowaniach zamiast naszego prostego kodu używa się:
- **Wektorowych baz danych** (Pinecone, Weaviate, ChromaDB, FAISS) zamiast numpy
- **Frameworków** (LangChain, LlamaIndex) do orkiestracji
- **Lepszych strategii chunkingu** (recursive, semantic chunking)
- **Re-rankingu** (dodatkowy model który ponownie ocenia trafność)
- **Hybrydowego wyszukiwania** (embeddingi + klasyczny BM25)
- **Multi-query RAG** (LLM generuje kilka wariantów pytania → lepsze pokrycie)

Ale **zasada jest dokładnie taka sama** jak to co zrobiliśmy na tych zajęciach!

### Kiedy RAG, a kiedy fine-tuning?

| | RAG | Fine-tuning (SFT) |
|---|---|---|
| **Kiedy?** | Dane się zmieniają, trzeba cytować źródła | Chcemy zmienić "styl" lub "wiedzę" modelu |
| **Koszt** | Tani (tylko inference) | Drogi (trening na GPU) |
| **Aktualność** | Zawsze aktualne (aktualizujesz dokumenty) | Wymaga ponownego treningu |
| **Halucynacje** | Mniejsze (model ma kontekst) | Możliwe (model "pamięta" błędnie) |